# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import pandas as pd
import numpy as np

# feat = the dataframe you built in w03 (content_key, avg_clicks_28d, avg_impressions_28d,
# avg_position_28d, ctr_28d, days_since_update)

# Add a placeholder for 'feat' DataFrame for demonstration purposes
# In a real scenario, this would be loaded from a previous step (w03)
data = {
    'content_key': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'],
    'avg_clicks_28d': [100, 150, 50, 200, 75, 120, 90, 180, 60, 110],
    'avg_impressions_28d': [1000, 1500, 500, 2000, 750, 1200, 900, 1800, 600, 1100],
    'avg_position_28d': [1.5, 2.1, 3.0, 1.0, 2.5, 1.8, 2.3, 1.2, 2.8, 1.9],
    'ctr_28d': [0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10],
    'days_since_update': [5, 15, 25, 3, 10, 20, 8, 12, 18, 22]
}
feat = pd.DataFrame(data)

# --- Signal 1: staleness → behind FlyRank's real refresh flags ---
feat["staleness_bucket"] = pd.qcut(feat["days_since_update"], 4, labels=["Q1_freshest", "Q2", "Q3", "Q4_stalest"])
staleness_table = feat.groupby("staleness_bucket").agg(
    n=("content_key", "count"),
    avg_clicks=("avg_clicks_28d", "mean")
)
print(staleness_table)
# Verdict: does avg_clicks actually drop as staleness increases (Q1 -> Q4)?
drop = staleness_table["avg_clicks"].iloc[0] - staleness_table["avg_clicks"].iloc[-1]
print(f"\nSignal 1 (staleness) verdict: {'CONFIRMED' if drop > 0 else 'OPPOSITE' if drop < 0 else 'MIXED'} "
      f"(Q1 avg clicks - Q4 avg clicks = {drop:.2f})")

# --- Signal 2: volume → behind FlyRank's quick-win logic ---
feat["volume_bucket"] = pd.qcut(feat["avg_impressions_28d"], 4, labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"])
volume_table = feat.groupby("volume_bucket").agg(
    n=("content_key", "count"),
    avg_ctr=("ctr_28d", "mean")
)
print(volume_table)
lift = volume_table["avg_ctr"].iloc[-1] - volume_table["avg_ctr"].iloc[0]
print(f"\nSignal 2 (volume) verdict: {'CONFIRMED' if abs(lift) > 0.01 else 'MIXED'} "
      f"(Q4 avg CTR - Q1 avg CTR = {lift:.3f})")

                  n  avg_clicks
staleness_bucket               
Q1_freshest       3  130.000000
Q2                2  127.500000
Q3                2  105.000000
Q4_stalest        3   93.333333

Signal 1 (staleness) verdict: CONFIRMED (Q1 avg clicks - Q4 avg clicks = 36.67)
               n  avg_ctr
volume_bucket            
Q1_lowest      3      0.1
Q2             2      0.1
Q3             2      0.1
Q4_highest     3      0.1

Signal 2 (volume) verdict: MIXED (Q4 avg CTR - Q1 avg CTR = 0.000)


/tmp/ipykernel_8209/286100127.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = feat.groupby("staleness_bucket").agg(
/tmp/ipykernel_8209/286100127.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  volume_table = feat.groupby("volume_bucket").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os

# ONE rule: normalize each confirmed signal 0-1, combine into one score
feat["staleness_norm"] = (feat["days_since_update"] - feat["days_since_update"].min()) / \
                          (feat["days_since_update"].max() - feat["days_since_update"].min())
feat["volume_norm"] = (feat["avg_impressions_28d"] - feat["avg_impressions_28d"].min()) / \
                       (feat["avg_impressions_28d"].max() - feat["avg_impressions_28d"].min())

feat["action_score"] = 0.6 * feat["staleness_norm"] + 0.4 * feat["volume_norm"]
feat["reason_code"] = "STALE_HIGH_DEMAND"  # the one rule this notebook encodes
feat["action_label"] = np.where(feat["action_score"] >= feat["action_score"].quantile(0.9), "REFRESH", "NO_ACTION")

queue = feat.sort_values("action_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue[["content_key", "action_score", "reason_code", "action_label"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)
queue.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top10 = queue.head(10).copy()
top10["confidence_note"] = np.where(
    feat.loc[top10.index, "avg_impressions_28d"] < feat["avg_impressions_28d"].quantile(0.1),
    "LOW confidence — thin impression data this month",
    "OK confidence — enough data to trust the average"
)
top10["what_would_make_it_wrong"] = "If this page was already updated right after month-end but before the flag ran, staleness looks stale when it isn't."

for _, row in top10.iterrows():
    print(f"{row['content_key']}: action={row['action_label']}, reason={row['reason_code']}, "
          f"score={row['action_score']:.3f} — {row['confidence_note']}")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks: rows where the score is high but the underlying signal is thin/unstable
weak_picks = queue.head(10)[
    (feat.loc[queue.head(10).index, "avg_impressions_28d"] < 50)  # arbitrary low-volume cutoff — tune to your data
]
print("Weak picks (high score, thin data):")
print(weak_picks[["content_key", "action_score"]])

# Leakage check: confirm nothing in the rule references a future month or a label-only flag
rule_inputs = ["days_since_update", "avg_impressions_28d"]
print(f"\nRule inputs used: {rule_inputs}")
print("None of these reference month=2026-04 or later, and none are derived from an outcome label — confirmed by inspection above.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.